# SecureLog Engine — Rapport synthétique

**Projet ING3 — Python 3.10+**  
**Membres du trinôme : Lukas MAUFFRÉ, Sacha BEHLOULI-DROLLÉE, Jeanne TONDA**

Nous avons choisi d'analyser les logs fictifs d'une salle informatique. Le
programme lit chaque ligne, vérifie son contenu puis met à jour les statistiques.
Nos trois règles recherchent les échecs de connexion consécutifs, les IP de la
blacklist et les événements critiques.

Avec nos fichiers de démonstration, nous obtenons **22 événements valides**,
**3 lignes rejetées** et **2 détections de force brute**. Le code principal se
trouve dans les fichiers `.py`. Ce Notebook sert à montrer les résultats.

## Architecture et choix

`main.py → ingestion → parser → processing → analyzer + rules → résultats + audit`

- `ingestion.py` lit les fichiers avec `yield` et `yield from`.
- `parser.py` transforme les lignes en dictionnaires et vérifie leurs champs.
- `processing.py` utilise `filter`, `lambda`, `map` et `match-case`.
- `analyzer.py` calcule les compteurs et les ensembles d'IP.
- `rules.py` contient les règles de sécurité et la détection de force brute.
- `audit.py` utilise `globals()` et `locals()` pour décrire les variables.

Nous avons séparé les étapes pour éviter d'avoir tout le code dans `main.py`.
Aucune bibliothèque externe n'est nécessaire.

In [1]:
from pathlib import Path
import json
import sys
import tempfile

ROOT = Path.cwd()
if ROOT.name == "report":
    ROOT = ROOT.parent
if not (ROOT / "main.py").is_file():
    raise RuntimeError("Ouvrir le Notebook depuis le projet ou son dossier report.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from main import run

with tempfile.TemporaryDirectory() as directory:
    output = Path(directory)
    summary = run(
        [ROOT / "data/auth.log", ROOT / "data/network.log"],
        ROOT / "data/blacklist.txt", output,
    )
    # Cette petite liste sert uniquement à illustrer les 15 alertes de démo.
    alerts = [json.loads(line) for line in
              (output / "alerts.jsonl").read_text().splitlines()]
    audit = json.loads((output / "audit.json").read_text())

print("Entrées :", summary["input"])
print("Alertes :", summary["alerts"])

Entrées : {'read': 28, 'ignored': 3, 'invalid': 3, 'valid': 22}
Alertes : {'BLACKLIST': 12, 'BRUTE_FORCE': 2, 'CRITICAL': 1}


## Résultats par niveau

Les trois rejets sont intentionnels : mauvais nombre de champs, IP invalide et
JSON tronqué. Ils permettent de montrer la gestion d'exceptions. Les commentaires
et les lignes vides sont ignorés séparément. Une erreur de sévérité ERROR ne
signifie pas nécessairement une attaque : elle contribue aux statistiques.

In [2]:
for level, count in summary["by_level"].items():
    print(f"{level:8} {'█' * count} {count}")
print("IP actives :", len(summary["active_ips"]))
print("IP ayant généré ERROR ou CRITICAL :", summary["error_ips"])

CRITICAL █ 1
ERROR    ███████████ 11
INFO     █████ 5
WARNING  █████ 5
IP actives : 6
IP ayant généré ERROR ou CRITICAL : ['192.0.2.30', '198.51.100.24', '203.0.113.66']


## Ensembles et détections

La blacklist est un `frozenset` : les règles ne peuvent pas la modifier.
L'intersection désigne les IP actives blacklistées ; la différence les IP
actives non blacklistées. L'union rassemble les deux ensembles et la différence
symétrique contient les IP présentes dans un seul des deux ensembles.

In [3]:
for operation, addresses in summary["sets"].items():
    print(f"{operation} : {addresses}")
print("\nBrute force :")
for alert in alerts:
    if alert["rule"] == "BRUTE_FORCE":
        print(f"{alert['ip']} / {alert['user']} : {alert['message']}")

union : ['192.0.2.10', '192.0.2.20', '192.0.2.250', '192.0.2.30', '192.0.2.40', '198.51.100.24', '203.0.113.66']
intersection : ['198.51.100.24', '203.0.113.66']
difference : ['192.0.2.10', '192.0.2.20', '192.0.2.30', '192.0.2.40']
symmetric_difference : ['192.0.2.10', '192.0.2.20', '192.0.2.250', '192.0.2.30', '192.0.2.40']

Brute force :
203.0.113.66 / admin : 5 échecs consécutifs
198.51.100.24 / root : 5 échecs consécutifs


Les cinq échecs d'admin et de root déclenchent chacun une alerte. Bob a deux
échecs, un succès, puis deux échecs : aucune série de cinq n'existe. Les identités
`(IP, utilisateur)` sont séparées ; les événements réseau n'interrompent pas
les essais d'authentification. Chaque bloc disjoint de cinq échecs produit une
alerte : dix échecs produisent deux alertes.

Les fichiers sont lus dans l'ordre fourni, sans tri global. Les lignes rejetées
ne peuvent pas participer à la détection. Les alertes désignent des suspicions,
à confirmer en examinant les événements et leur provenance.

In [4]:
for name in ("blacklist", "counters", "histories", "analyzer"):
    print(name, audit["pipeline_locals"][name])
print("Module audité :", audit["audit_globals"]["AUDIT_VERSION"])

blacklist {'type': 'frozenset', 'size': 3}
counters {'type': 'dict', 'size': 4}
histories {'type': 'dict', 'size': 1}
analyzer {'type': 'Analyzer'}
Module audité : {'type': 'str', 'size': 3}


## Validation et consommation mémoire

Nous avons écrit **22 tests automatisés**. Ils vérifient le parsing, les lignes
incorrectes, la blacklist, les seuils, les opérations sur les ensembles et
l'exécution complète. La commande est :
`python -m unittest discover -s tests -v`.

Les 22 tests réussissent. Le projet demande Python 3.10 ou une version plus
récente et respecte les principales règles de présentation PEP8.

Nous avons aussi essayé le programme avec 10 000 puis 100 000 lignes. Dans cet
essai, une seule IP et un seul utilisateur sont utilisés. La mémoire a été
mesurée avec le module `tracemalloc`.

| Lignes | Taille du fichier | Pic des allocations Python | Temps observé |
|---:|---:|---:|---:|
| 10,000 | 601,000 octets | 308,007 octets | 0.227 s |
| 100,000 | 6,010,000 octets | 301,007 octets | 1.803 s |

Le nombre de lignes est multiplié par dix mais le pic reste proche de 300 000
octets. Cela confirme que les lignes ne sont pas toutes gardées en mémoire.
Les temps et les mesures peuvent changer selon l'ordinateur utilisé.

Le programme doit quand même conserver les IP et utilisateurs différents pour
calculer ses statistiques. La mémoire peut donc augmenter si leur nombre augmente.

## Limites et conclusion

Notre programme analyse des fichiers déjà créés. Il ne surveille pas les nouvelles
lignes en continu et ne bloque aucune IP. Les fichiers sont lus dans l'ordre donné.

Cette version répond aux notions demandées dans l'énoncé. Si nous devions continuer
le projet, nous pourrions ajouter une limite de temps à la détection de force brute.

## Source du cahier des charges

ÉNONCÉ DU PROJET FINAL : ENGINE DE TRAITEMENT ET D'ANALYSE DE FLUX DE DONNÉES
SÉCURISÉ EN TEMPS RÉEL, niveau ING3, document fourni par l'enseignant.